# Урок 7. Улучшение модели матчинга

FE: fuzzy flat number, калибровка порога, очередь на проверку, разбор ошибок.

In [1]:
from pathlib import Path
import sys, pandas as pd, numpy as np
from rapidfuzz import fuzz
from sklearn.metrics import precision_recall_curve
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from matching_service.application.train import prepare_xy, FEATURE_COLS
from catboost import CatBoostClassifier
holdout = pd.read_parquet(ROOT/'artifacts/datasets/tyumen_pairs_holdout.parquet')
train = pd.read_parquet(ROOT/'artifacts/datasets/tyumen_pairs_train.parquet')

def fuzzy_flat(a,b):
    a=str(a or ''); b=str(b or '')
    if not a or not b: return 0.0
    return fuzz.ratio(a,b)/100.0

for df in (train, holdout):
    df['fuzzy_flat']= [fuzzy_flat(a,b) for a,b in zip(df['flat_number_deal'], df['flat_number_exp'])]
feats = FEATURE_COLS + ['fuzzy_flat']
Xtr,ytr = train[feats].apply(pd.to_numeric, errors='coerce').fillna(0), train['label']
Xho,yho = holdout[feats].apply(pd.to_numeric, errors='coerce').fillna(0), holdout['label']
model = CatBoostClassifier(depth=6, iterations=400, learning_rate=0.07, auto_class_weights='Balanced', verbose=False, random_seed=42)
model.fit(Xtr,ytr)
prob = model.predict_proba(Xho)[ :,1]
prec, rec, thr = precision_recall_curve(yho, prob)
f1 = 2*prec*rec/(prec+rec+1e-9)
best = int(np.nanargmax(f1))
print('best_threshold', thr[best] if best < len(thr) else 0.5, 'precision', prec[best], 'recall', rec[best], 'f1', f1[best])
review = holdout.iloc[((prob > thr[best]-0.08) & (prob < thr[best]+0.05)).nonzero()[0] if False else np.where((prob>0.4)&(prob<0.6))[0]]
print('review_queue', len(review))
model.save_model(str(ROOT/'artifacts/models/catboost_match_v2.cbm'))

best_threshold 0.825077693394612 precision 0.8717893466643682 recall 0.8695783003052057 f1 0.8706824192817815
review_queue 2459


## Улучшенная архитектура: эмбеддинги + реранкер в каскаде

Полный каскад (`src/matching_service/application/cascade.py`):

`rules → CatBoost → embeddings (BGE-M3) → reranker (BGE-reranker-v2-m3) → LiteLLM (ambiguous)`

- **Embeddings (BGE-M3)** — семантический скор пары «описание сделки ↔ объявление», подтягивается с HuggingFace (`HF_TOKEN`), lazy-load.
- **Reranker (BGE-reranker-v2-m3)** — cross-encoder для переранжирования кандидатов; `reranker_score` входит в ensemble с весом ~0.2 и в precision-first проверку top-1.
- Веса: `MATCH_EMBEDDING_MODEL`, `MATCH_RERANKER_MODEL`; флаги `MATCH_USE_EMBEDDINGS`, `MATCH_USE_RERANKER`.

## Выводы
- fuzzy номер квартиры снижает ложные отрицания при форматных расхождениях;
- порог выбираем по max F1 на hold-out;
- спорные скоры → очередь ручной проверки (Precision-first).